# Claude に決まった形式 (JSON スキーマ) で出力させる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_llm/claude_structured.ipynb)

NLP クックブック 12-12 ／ 大規模言語モデル (LLM)

> API キーが必要 (Colab 左の 🔑 シークレットに登録)

テキストから情報を抽出して、型の決まったデータとして受け取りたい。JSON のパース失敗に悩まされたくない。

In [ ]:
!pip install -q anthropic pydantic

In [ ]:
from typing import Literal
import anthropic
from google.colab import userdata
from pydantic import BaseModel

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

class Entity(BaseModel):
    text: str
    type: Literal["人名", "組織名", "地名", "日付", "金額"]

class Extraction(BaseModel):
    entities: list[Entity]
    summary: str
    sentiment: Literal["positive", "negative", "neutral"]

text = "2024年4月、山田太郎氏はトヨタ自動車を退職し、京都市でスタートアップを創業した。創業半年で3億円を調達し、順調な滑り出しを見せている。"
response = client.messages.parse(
    model="claude-opus-5-5",
    max_tokens=16000,
    messages=[{"role": "user", "content": f"次の文章から固有表現を抽出し、一文で要約し、論調を判定してください。\n\n{text}"}],
    output_format=Extraction,
)

if response.stop_reason == "refusal":
    print("応答が拒否されました")
else:
    result = response.parsed_output          # 検証済みの Extraction インスタンス
    for e in result.entities:
        print(e.type, e.text)
    print(result.summary, "/", result.sentiment)

- 構造化出力では、スキーマに従う JSON しか生成されないように制約される。`Literal` で選択肢を限定すれば、分類タスクにもそのまま使える。
- 抽出した `text` が原文に実在するかは保証されない。`e.text in text` で検証するとよい。文字位置が必要なら、自分で `text.find()` する。
- スキーマに書けない制約 (最小値・最大値など) は、Pydantic 側の検証で確認する。